In [1]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np

# Load data
data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

In [2]:
from sklearn.model_selection import train_test_split
import xgboost as xgb

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

In [3]:
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': 42
}
model = xgb.train(params, dtrain, num_boost_round=100)

In [4]:
dval = xgb.DMatrix(X_test, label=y_test)
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=[(dval, 'validation')],
    early_stopping_rounds=10,
    verbose_eval=50
)

[0]	validation-rmse:0.93842
[50]	validation-rmse:0.48511
[100]	validation-rmse:0.47195
[145]	validation-rmse:0.46973


In [5]:
# xgb.plot_importance(model)
# xgb.plot_tree(model, num_trees=0)

In [6]:
from sklearn.metrics import mean_squared_error, r2_score

preds = model.predict(dtest)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"RMSE: {rmse:.4f}, R²: {r2:.4f}")

RMSE: 0.4697, R²: 0.8316


In [7]:
model.save_model("xgb_california_model.json")
model_loaded = xgb.Booster()
model_loaded.load_model("xgb_california_model.json")

In [8]:
params = {
    'objective': 'reg:squarederror',
    'monotone_constraints': "(0,0,1,0,0,0,0,0)"
}


In [9]:
def custom_rmse(preds, dtrain):
    labels = dtrain.get_label()
    return 'custom_rmse', np.sqrt(((preds - labels)**2).mean())

model = xgb.train(params, dtrain, num_boost_round=100, feval=custom_rmse)

TypeError: train() got an unexpected keyword argument 'feval'

In [10]:
leaf_indices = model.predict(dtest, pred_leaf=True)
print(leaf_indices[:5])

[[ 65.  78.  72.  86.  77.  92.  85.  76.  78.  85.  79. 119.  84.  58.
  104.  83.  67.  74.  84.  80.  96.  80.  80.  72.  80.  88.  68.  77.
   70.  70. 120. 117.  68.  92.  91.  89.  90.  68.  90.  68.  75.  65.
  100. 103. 122.  74.  79.  59. 100.  70. 100.  61.  83.  84.  72.  98.
   68.  69.  66.  92.  77.  75.  66.  93.  60.  57. 121.  72.  64.  68.
   70.  70.  62.  66.  85.  85.  73.  78. 110.  66. 101.  90.  90.  92.
   57. 111. 111. 113.  63. 115. 108.  62. 108.  55.  98.  76.  86.  89.
   86.  76. 119.  88.  62.  98. 111.  63.  67. 104.  75.  65.  88. 103.
   97.  80.  77.  63.  61. 115.  76.  78.  53.  72. 101.  67.  99.  61.
   85. 111. 102.  78. 118.  63.  75.  60.  63. 106.  58.  99.  90.  93.
   74.  77.  98.  87.  54.  63.]
 [ 78.  78.  75.  90.  77.  92.  70.  75.  89.  76.  79.  90.  84.  58.
  104.  94.  67.  74.  84.  80.  96.  79.  80.  72.  95.  92.  68.  77.
   70.  70. 118. 117.  68.  96.  92. 114.  87.  68. 105.  68.  75.  88.
   98. 103. 122.  86.  79.  59.